# Recorded-episode latent diagnostics
Standalone diagnostics for the split-camera action64 world-model checkpoints (v4, no actor). Encodes **every recorded observation** (both cameras and proprioception) in an eval demonstration; this is not a simulated or predicted rollout. No MuJoCo, actor, CEM, dataset download, or manual subgoal labels are needed.

The goal is the final stored observation. PCA is fitted independently to each selected episode; axes are not comparable between separate episode plots. Distance is computed in the original 192D latent space. A zero final MSE is guaranteed because the final embedding is also the goal—not evidence of model accuracy.


In [ ]:
%pip install -q h5py matplotlib numpy pillow tqdm


## Google Drive setup
Use `MyDrive/primary_split_new/evals.hdf5` and `MyDrive/primary_split_new/models/best_model.zip` (or `epoch_N_model.zip`). Outputs go to `primary_split_new/diagnostics/`.
Mount Drive below, then run the model-definition cell once. Change the controls in the final cell and rerun **only that cell**. Only load checkpoints you trust. A T4-class GPU is sufficient for this encoder-only diagnostic; CPU also works, more slowly.


In [ ]:
import io
import json
import re
import zipfile
from pathlib import Path
import h5py
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def normalize_robot_images(images):
    shape = [1] * images.ndim
    shape[-3] = 3
    mean = images.new_tensor((0.485, 0.456, 0.406)).reshape(shape)
    std = images.new_tensor((0.229, 0.224, 0.225)).reshape(shape)
    return (images - mean) / std

def build_robot_state(pos, quat, grip):
    quat = quat / quat.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return torch.cat((pos, quat, grip), dim=-1)


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn


class ProjectionMLP(nn.Module):
    """LeWM projection head used after encoder and predictor features."""

    def __init__(self, input_dim: int = 192, hidden_dim: int = 2048, output_dim: int = 192):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.layers(x)


class ActionEncoder(nn.Module):
    """Encode one ordered primitive-action block for AdaLN conditioning."""

    def __init__(self, input_dim: int, smoothed_dim: int = 10, embedding_dim: int = 192):
        super().__init__()
        self.smoother = nn.Conv1d(input_dim, smoothed_dim, kernel_size=1)
        self.mlp = nn.Sequential(
            nn.Linear(smoothed_dim, 4 * embedding_dim),
            nn.SiLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )

    def forward(self, actions):
        batch_size, sequence_length, _ = actions.shape
        actions = self.smoother(actions.transpose(1, 2)).transpose(1, 2)
        return self.mlp(actions.reshape(batch_size * sequence_length, -1)).reshape(
            batch_size, sequence_length, -1
        )


class TransformerBlock(nn.Module):
    """Pre-norm transformer block with optional AdaLN-Zero conditioning."""

    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        d_action: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
        adaptive: bool = True,
        dim_head: int | None = None,
        mlp_dim: int | None = None,
    ):
        super().__init__()
        self.adaptive = adaptive
        hidden_dim = mlp_dim or 4 * d_model

        self.norm_one = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        self.norm_two = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        if adaptive:
            if d_action is None:
                raise ValueError("d_action is required for an adaptive transformer block")
            self.adaLN_modulation = nn.Sequential(
                nn.SiLU(),
                nn.Linear(d_action, 6 * d_model),
            )
            nn.init.zeros_(self.adaLN_modulation[-1].weight)
            nn.init.zeros_(self.adaLN_modulation[-1].bias)

        self.attn_block = MultiHeadAttention(
            d_model=d_model,
            nheads=nheads,
            dim_head=dim_head,
            dropout=dropout,
            causal_masking=causal_masking,
        )
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout),
        )

    @staticmethod
    def _modulate(x, shift, scale):
        return x * (1 + scale) + shift

    def forward(self, z, a=None):
        if self.adaptive:
            if a is None:
                raise ValueError("action conditioning is required for an adaptive block")
            shift_attn, scale_attn, gate_attn, shift_mlp, scale_mlp, gate_mlp = (
                self.adaLN_modulation(a).chunk(6, dim=-1)
            )
            z = z + gate_attn * self.attn_block(
                self._modulate(self.norm_one(z), shift_attn, scale_attn)
            )
            z = z + gate_mlp * self.mlp(
                self._modulate(self.norm_two(z), shift_mlp, scale_mlp)
            )
        else:
            z = z + self.attn_block(self.norm_one(z))
            z = z + self.mlp(self.norm_two(z))
        return z


class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        dim_head: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_k = dim_head or (d_model // nheads)
        self.nheads = nheads
        self.inner_dim = self.nheads * self.d_k
        self.causal_masking = causal_masking

        self.W_q = nn.Linear(d_model, self.inner_dim)
        self.W_k = nn.Linear(d_model, self.inner_dim)
        self.W_v = nn.Linear(d_model, self.inner_dim)
        self.W_o = nn.Linear(self.inner_dim, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        queries = self.W_q(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        keys = self.W_k(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        values = self.W_v(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)

        attention = F.scaled_dot_product_attention(
            queries,
            keys,
            values,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=self.causal_masking,
        )
        attention = attention.transpose(1, 2).reshape(
            batch_size, sequence_length, self.inner_dim
        )
        return self.drop(self.W_o(attention))

from torch import nn
import torch

"""
Thought it would be fun to define the architecture of the ViT-Tiny
from scratch so here it is using the transformers I defined as well
"""

class CustomVitTiny(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256):
        super().__init__()

        self.npatches = npatches
        self.d_model = d_model

        self.conv_flattener = nn.Conv2d(in_channels=3,
                                        out_channels=d_model,
                                        kernel_size=patch,
                                        stride=patch,
                                        padding=0)
        
        self.transformer_layers = nn.ModuleList([])
        for _ in range(nlayers):
            self.transformer_layers.append(TransformerBlock(d_model=d_model, 
                                           nheads=nheads,                                            
                                           dropout=0.0,
                                           causal_masking=False,
                                           adaptive=False))
        # learned positional embeddings
        self.position_emb = nn.Parameter(torch.empty(npatches + 1, d_model))
        # learned [cls] embedding
        self.cls_emb = nn.Parameter(torch.empty(1, 1, 1, d_model))
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.position_emb, std=0.02)
        nn.init.trunc_normal_(self.cls_emb, std=0.02)

    def forward(self, x):
        # shape of x: (B, N, C, H, W)
        B, N, C, H, W = x.shape
        x = x.reshape(B*N, C, H, W)
        emb = self.conv_flattener(x)
        emb = emb.reshape(B, N, self.d_model, self.npatches).permute(0, 1, 3, 2)
        # shape of emb now: (512, 3, 256, 192)
        cls_emb = self.cls_emb.expand(B, N, 1, self.d_model)
        emb = torch.cat([cls_emb, emb], dim=2)
        emb = emb + self.position_emb
        # shape of emb now: (512, 3, 257, 192)
        emb = emb.reshape(B*N, self.npatches + 1, self.d_model) # now ready to go into the transformer

        for layer in (self.transformer_layers):
           emb = layer(emb)
        emb = self.norm(emb)

        emb = emb.reshape(B, N, self.npatches + 1, self.d_model)
        emb = emb[:, :, 0, :] # shape : (512, 3, 192), this is what the predictor expects as the input

        return emb





# TEST CODE for Shape
# dummy_conv = nn.Conv2d(in_channels=3,
#           out_channels=192,
#           kernel_size=14,
#           stride=14,
#           padding=0)

# dummy_frame = torch.ones((512, 3, 3, 224, 224))
# dummy_frame = dummy_frame.reshape(512*3, 3, 224, 224)
# emb = dummy_conv(dummy_frame)

# print(emb.shape)
# print("after reshaping")
# emb = emb.reshape((512, 3, 192, 256))
# emb = emb.permute(0, 1, 3, 2)
# print(emb.shape)
# cls = torch.ones(1, 1, 1, 192)
# cls = cls.expand(512, 3, 1, cls.shape[-1])
# emb = torch.cat([cls, emb], dim=2)
# print(emb.shape)
# emb = emb[:, :, 0, :]
# print(emb.shape)

# TEST CODE for the Architecture
# test = CustomVitTiny()
# dummy = torch.ones(2, 3, 3, 224, 224)
# dummy = test(dummy)
# print(dummy.shape)

from torch import nn
import torch

"""
This is the network definition for the encoder network which in the paper
was just ViT-Tiny which is a visual transformer with the following parameters:
- patch size of 14, 12 layers, 3 attention heads, and a hidden dimension of 192
- there is a "projection" layer added to the end of this because visual transformers
  typically have layernorms at the end but this ruins the regularization used to
  prevent representation collapse
    - this projection layer is a single layer with batch normalization to undo the
      affects of the layernorm at the end
"""
# right now the image size is defaulted to 224x224

class EncoderNetwork(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256,
                 projector_hidden_dim: int = 2048,
                 project_output: bool = True): # npatches is (H/patch_size)*(W/patch_size)
        super().__init__()
        self.d_model = d_model
        
        self.vit = CustomVitTiny(patch, nheads, nlayers, d_model, npatches)
        self.projector = (
            ProjectionMLP(d_model, projector_hidden_dim, d_model)
            if project_output else nn.Identity()
        )
    
    def forward(self, x):
        # shape of input (x): (B, N, C, H, W)
        B, N, C, H, W = x.shape
        z = self.vit(x)
        #projection step
        z = z.reshape(B*N, self.d_model)
        z = self.projector(z)
        z = z.reshape(B, N, self.d_model)

        return z
    

# TEST Code
# dummy_image = torch.ones(24, 3, 3, 224, 224)
# enc = EncoderNetwork()
# print(enc(dummy_image).shape)

from torch import nn
import torch




"""
This is the modified encoder network that extends it to take in
2 camera observations plus robot state information and then output
a final embedding in latent space. Also modifying it to be 84x84 to match
the mimicgen dataset instead of 224x224 which was for the original definition.
This encoder will now seperately pass through each input image through
the default encoder and then concatenate them together with the robot state
embedding and then this concatenated vector will be passed through a fusion MLP
to compress it back to the original embedding space. The output of this encoder
will still be the standard dimension so it can be passed through the default
predictor
"""


class RobotEncoder(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 d_state : int = 9,
                 d_state_emb : int = 64,
                 npatches : int = 36,
                 projector_hidden_dim: int = 2048):  # 84x84, patch 14 → 6x6 patches
        super().__init__()
        encoder_kwargs = dict(
            patch=patch,
            nheads=nheads,
            nlayers=nlayers,
            d_model=d_model,
            npatches=npatches,
            projector_hidden_dim=projector_hidden_dim,
            project_output=False,
        )
        # The architectures are identical, but no parameters are shared. This
        # lets each visual backbone specialize to its camera's representation.
        self.agent_encoder = EncoderNetwork(**encoder_kwargs)
        self.wrist_encoder = EncoderNetwork(**encoder_kwargs)

        # small projection network for the 
        self.state_encoder = nn.Sequential(
            nn.Linear(in_features=d_state, out_features=2*d_state),
            nn.GELU(),
            nn.Linear(in_features=2*d_state, out_features=d_state_emb)
        )

        self.fusion_mlp = nn.Sequential(
            nn.Linear(in_features=(d_model * 2 + d_state_emb), out_features=(d_model * 2)),
            nn.GELU(),
            nn.Linear(in_features=(d_model * 2), out_features=d_model),
        )
        self.projector = ProjectionMLP(d_model, projector_hidden_dim, d_model)

    def forward(self, above, wrist, state):
        # above : [Batch, Window, 3, 84, 84] | wrist : [Batch, Window, 3, 84, 84] | state : [Batch, Window, 9]
        above_emb = self.agent_encoder(above)
        wrist_emb = self.wrist_encoder(wrist)
        state_emb = self.state_encoder(state) # [Batch, Window, 64]
        fusion = torch.cat((above_emb, wrist_emb, state_emb), dim=-1) # [Batch, Window, 448]
        fusion = self.fusion_mlp(fusion)
        flat = fusion.reshape(-1, fusion.shape[-1])
        return self.projector(flat).reshape_as(fusion) # [Batch, Window, 192]

import torch
from torch import nn







In [ ]:
def load_encoder(checkpoint, device):
    with zipfile.ZipFile(checkpoint) as archive:
        stats = torch.load(io.BytesIO(archive.read('state_stats.pth')), map_location='cpu', weights_only=True)
        expected = {
            'checkpoint_format_version': 4,
            'architecture_id': 'lewm_robot_dual_encoder_action64_v4',
            'image_normalization_id': 'imagenet',
        }
        for key, value in expected.items():
            if stats.get(key) != value:
                raise ValueError(f'Incompatible checkpoint: {key}={stats.get(key)!r}; expected {value!r}')
        model = RobotEncoder()
        weights = torch.load(io.BytesIO(archive.read('encoder_weights.pth')), map_location='cpu', weights_only=True)
        model.load_state_dict(weights, strict=True)
    mean = torch.as_tensor(stats['state_mean'], dtype=torch.float32, device=device)
    std = torch.as_tensor(stats['state_std'], dtype=torch.float32, device=device)
    if mean.shape != (9,) or std.shape != (9,) or not torch.isfinite(mean).all() or not torch.isfinite(std).all() or (std <= 0).any():
        raise ValueError('Invalid proprioception normalization statistics')
    return model.to(device).eval(), mean, std

@torch.inference_mode()
def encode_episode(model, obs, mean, std, batch_size, device):
    keys = ['agentview_image', 'robot0_eye_in_hand_image',
            'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos']
    n = len(obs[keys[0]])
    if n < 2 or any(len(obs[key]) != n for key in keys):
        raise ValueError('Need at least two aligned observations')
    outputs = []
    for start in tqdm(range(0, n, batch_size), desc='Encoding all recorded frames'):
        stop = min(start + batch_size, n)
        images = []
        for key in keys[:2]:
            raw = obs[key][start:stop]
            if raw.shape[1:] != (84, 84, 3) or raw.dtype != np.uint8:
                raise ValueError(f'{key}: expected uint8 [N,84,84,3], got {raw.shape}, {raw.dtype}')
            x = torch.as_tensor(raw, device=device).permute(0, 3, 1, 2).float() / 255.
            images.append(normalize_robot_images(x)[:, None])
        parts = [torch.as_tensor(obs[key][start:stop], dtype=torch.float32, device=device) for key in keys[2:]]
        state = (build_robot_state(*parts) - mean) / std
        outputs.append(model(images[0], images[1], state[:, None])[:, 0].float().cpu().numpy())
    z = np.concatenate(outputs)
    if not np.isfinite(z).all():
        raise ValueError('Encoder produced nonfinite embeddings')
    return z

def trajectory_metrics(z):
    centered = z.astype(np.float64) - z.mean(axis=0, dtype=np.float64)
    _, singular, vt = np.linalg.svd(centered, full_matrices=False)
    xy = centered @ vt[:2].T
    variance = singular ** 2
    explained = variance[:2] / variance.sum() if variance.sum() > 0 else np.zeros(2)
    mse = ((z.astype(np.float64) - z[-1]) ** 2).mean(axis=1)
    return xy, explained, mse


## Select an episode and generate plots
`EVAL_EPISODE=1` selects the first numerically sorted demo (usually `demo_0`), not necessarily `demo_1`. The actual demo name is printed. `None` uses best validation checkpoint; explicit epochs use zero-based checkpoint numbering. Saved plots are overwritten when rerunning the same checkpoint/episode combination.


In [ ]:
# ----------------------- diagnostic controls -----------------------
DRIVE_DIR = Path('/content/drive/MyDrive/primary_split_new')
EVAL_EPISODE = 1       # 1–50 for your 50-demo bank (one-based selection).
CHECKPOINT_EPOCH = None  # None: best_model.zip; otherwise e.g. 72.
ENCODE_BATCH_SIZE = 32   # Reduce to 8 if GPU memory is limited.
SAVE_PLOTS = True
# -------------------------------------------------------------------
if not isinstance(EVAL_EPISODE, int) or isinstance(EVAL_EPISODE, bool):
    raise ValueError('EVAL_EPISODE must be an integer')
if not isinstance(ENCODE_BATCH_SIZE, int) or ENCODE_BATCH_SIZE < 1:
    raise ValueError('ENCODE_BATCH_SIZE must be positive')
if CHECKPOINT_EPOCH is not None and (not isinstance(CHECKPOINT_EPOCH, int) or isinstance(CHECKPOINT_EPOCH, bool) or CHECKPOINT_EPOCH < 0):
    raise ValueError('CHECKPOINT_EPOCH must be None or a nonnegative integer')
checkpoint = DRIVE_DIR / 'models' / ('best_model.zip' if CHECKPOINT_EPOCH is None else f'epoch_{CHECKPOINT_EPOCH}_model.zip')
eval_path = DRIVE_DIR / 'evals.hdf5'
for path in (checkpoint, eval_path):
    if not path.is_file():
        raise FileNotFoundError(path)
# Reload when changing checkpoint; keep only one encoder resident on the GPU.
if 'encoder' in globals():
    del encoder
encoder, state_mean, state_std = load_encoder(checkpoint, DEVICE)
with h5py.File(eval_path, 'r') as f:
    names = sorted((name for name in f['data'] if re.fullmatch(r'demo_\d+', name)),
                   key=lambda name: int(name.split('_')[-1]))
    if not 1 <= EVAL_EPISODE <= len(names):
        raise ValueError(f'Choose EVAL_EPISODE between 1 and {len(names)}')
    demo_name = names[EVAL_EPISODE - 1]
    embeddings = encode_episode(encoder, f['data'][demo_name]['obs'],
                                state_mean, state_std, ENCODE_BATCH_SIZE, DEVICE)
xy, explained, mse = trajectory_metrics(embeddings)
steps = np.arange(len(embeddings))
print(f'{demo_name}: {len(steps)} frames; goal frame {steps[-1]}; checkpoint {checkpoint.name}')
print(f'PCA retained variance: {100 * explained.sum():.1f}%')
print(f'Latent MSE: start={mse[0]:.6f}, final={mse[-1]:.6f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(xy[:, 0], xy[:, 1], color='0.65', linewidth=1, zorder=1)
points = axes[0].scatter(xy[:, 0], xy[:, 1], c=steps, cmap='viridis', s=18, zorder=2)
axes[0].scatter(*xy[0], s=90, marker='o', facecolor='white', edgecolor='black', label='Start', zorder=3)
axes[0].scatter(*xy[-1], s=200, marker='*', color='gold', edgecolor='black', label='Final goal', zorder=4)
axes[0].set(xlabel=f'PC1 ({explained[0]:.1%})', ylabel=f'PC2 ({explained[1]:.1%})', title='PCA of encoded observations')
axes[0].legend()
fig.colorbar(points, ax=axes[0], label='Recorded timestep')
axes[1].plot(steps, mse, color='indianred')
axes[1].set(xlabel='Recorded timestep', ylabel='Mean squared latent distance to final goal', title='MSE in full 192D latent space')
axes[1].grid(alpha=0.2)
fig.suptitle(f'{demo_name} | {checkpoint.stem}')
if SAVE_PLOTS:
    output_dir = DRIVE_DIR / 'diagnostics'
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = output_dir / f'{demo_name}_{checkpoint.stem}'
    fig.savefig(str(stem) + '.png', dpi=220)
    fig.savefig(str(stem) + '.svg')
    np.savez_compressed(str(stem) + '.npz', embeddings=embeddings, pca=xy, mse=mse,
                        timesteps=steps, explained_variance_ratio=explained)
    print('Saved PNG, SVG and numerical data:', stem)
plt.show()
